In [77]:
%load_ext autoreload
%autoreload 2

import sys
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("/root/limlab01/kaistai/25DFT/QHFlow/src")
from md.scflow_calculator_gpu import SCFlowRKSCalculator, RKSCalculator
from dft_process.dft_process_utils import *
from common.draw_util import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import time

In [74]:
# md17_experiment = MD17Experiment(data_index=1)
# md17_experiment.set_model_path()
qh9_experiment = QH9Experiment(data_index=1, use_shard=True)

cur_ckpt = "/root/25DFT/QHFlow/src/outputs/QH9Stable-random/QHFlow_so2_v5_1_large-QH9Stable-random/checkpoints/weights-epoch=78-val_loss=0.0000000.ckpt"

In [44]:
qh9_experiment.dataset[0].dft_energy

tensor([[-40.4877]], dtype=torch.float64)

In [45]:
gt_outputs = qh9_experiment.dataset.get_gt_outputs(0)

In [46]:
gt_outputs

{'pos': tensor([[-1.2698e-02,  1.0858e+00,  8.0010e-03],
         [ 2.1504e-03, -6.0313e-03,  1.9761e-03],
         [ 1.0117e+00,  1.4638e+00,  2.7657e-04],
         [-5.4082e-01,  1.4475e+00, -8.7664e-01],
         [-5.2381e-01,  1.4379e+00,  9.0640e-01]], dtype=torch.float64),
 'atoms': tensor([[6],
         [1],
         [1],
         [1],
         [1]]),
 'hamiltonian': tensor([[-10.1627,  -3.5317,  -1.6758,  ...,  -0.4342,   0.2991,   0.7632],
         [ -3.5317,  -1.6945,  -1.1353,  ...,  -0.2141,   0.1475,   0.3763],
         [ -1.6758,  -1.1353,  -0.8022,  ...,  -0.1481,   0.1020,   0.2603],
         ...,
         [ -0.4342,  -0.2141,  -0.1481,  ...,   1.2700,   0.0483,   0.1233],
         [  0.2991,   0.1475,   0.1020,  ...,   0.0483,   1.3069,  -0.0850],
         [  0.7632,   0.3763,   0.2603,  ...,   0.1233,  -0.0850,   1.1234]],
        dtype=torch.float64),
 'overlap_matrix': tensor([[ 1.0000,  0.3256,  0.1518,  ...,  0.0387, -0.0267, -0.0680],
         [ 0.3256,  1.0000, 

In [47]:
# import Batch
from torch_geometric.data import Batch
qh9_d0 = Batch.from_data_list([qh9_experiment.dataset[0]])

In [75]:
sccalculator = SCFlowRKSCalculator(basis="def2-SVP", functional="b3lyp")
sccalculator.set_model(cur_ckpt, data_type="qh9", units="ang", model_length_unit="ang", mf_init_functional="b3lyp",)

INFO     [__init__.py:104] >> model_args: {'in_node_features': 1, 'sh_lmax': 4, 'hidden_size': 256,                
         'bottle_hidden_size': 64, 'num_gnn_layers': 4, 'max_radius': 15, 'num_nodes': 10, 'radius_embed_dim': 16, 
         'max_T': 15, 'use_block_S': True, 'use_block_H': True, 'ham_dim': 24, 'ham_hidden': 288, 'dataset_type':  
         'qh9', 'num_ham_gnn_layers': 2}

Setting device to cuda


INFO     [base_module.py:128] >> use_init_hamiltonian: True

INFO     [base_module.py:129] >> use_init_hamiltonian_residue: True

INFO     [base_module.py:130] >> ema_start_epoch: -1

INFO     [base_module.py:131] >> qh9: True

INFO     [base_module.py:132] >> test_mode: test

INFO     [flow_module.py:211] >> init_p0_type: expand

model trained on dataset:  QH9Stable
ode_steps: 1


In [ ]:
mol = sccalculator.get_mol(qh9_d0.atoms, qh9_d0.pos)
ase_atoms = md.scflow_calculator_gpu.QHData_to_atoms(qh9_d0, unit="ang")
res2 = sccalculator.calculate(ase_atoms, properties=["energy", "forces"], system_changes=["positions"])

In [99]:
rkscalculator = md.scflow_calculator_gpu.RKSCalculator(basis="def2-SVP", functional="b3lyp", density_fit=True)
res3 = rkscalculator.calculate(ase_atoms, properties=["energy, forces"], system_changes=["positions"])
gt_fock = rkscalculator.mf.get_fock()

In [ ]:
xyz_folder = "/root/25DFT/QHFlow/qhflow_md_yh/yh_dft_test/xyz_files"
file_lists = os.listdir(xyz_folder)

In [ ]:
# Algorithm to sort xyz files by atom size (number of atoms)

from ase.io import read

def get_atom_count(xyz_file_path):
    """Extract atom count from the first line of xyz file"""
    with open(xyz_file_path, 'r') as f:
        first_line = f.readline().strip()
        return int(first_line)

# Store filename, atom count, and file path as tuples
file_atom_pairs = []
for filename in file_lists:
    if filename.endswith('.xyz'):
        file_path = os.path.join(xyz_folder, filename)
        try:
            atom_count = get_atom_count(file_path)
            file_atom_pairs.append((filename, atom_count, file_path))
        except Exception as e:
            print(f"Error reading {filename}: {e}")

# Sort by atom count (ascending order)
sorted_files = sorted(file_atom_pairs, key=lambda x: x[1])

# Display sorted results
print(f"Total {len(sorted_files)} files sorted by atom count:\n")
for filename, atom_count, file_path in sorted_files:
    print(f"{filename}: {atom_count} atoms")

# Extract sorted file names and paths
sorted_file_names = [filename for filename, _, _ in sorted_files]
sorted_file_paths = [file_path for _, _, file_path in sorted_files]

# Load xyz files using ASE
loaded_xyz_structures = []
for file_path in sorted_file_paths:
    try:
        atoms = read(file_path)
        loaded_xyz_structures.append(atoms)
        print(f"Loaded: {os.path.basename(file_path)} - {len(atoms)} atoms")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

print(f"\nSuccessfully loaded {len(loaded_xyz_structures)} xyz files")

Total 60 files sorted by atom count:

177820666.xyz: 44 atoms
177820663.xyz: 45 atoms
177820662.xyz: 45 atoms
177820667.xyz: 48 atoms
177820659.xyz: 48 atoms
177820658.xyz: 48 atoms
177820673.xyz: 49 atoms
177820660.xyz: 51 atoms
177820661.xyz: 51 atoms
177820657.xyz: 51 atoms
177818754.xyz: 60 atoms
177818753.xyz: 60 atoms
177818591.xyz: 62 atoms
177818592.xyz: 62 atoms
177818577.xyz: 62 atoms
177818580.xyz: 62 atoms
177818176.xyz: 65 atoms
177820513.xyz: 74 atoms
177818402.xyz: 75 atoms
177818755.xyz: 75 atoms
177814930.xyz: 83 atoms
177814931.xyz: 83 atoms
177814854.xyz: 84 atoms
177818419.xyz: 87 atoms
177815370.xyz: 88 atoms
177814195.xyz: 89 atoms
177814855.xyz: 94 atoms
177818761.xyz: 96 atoms
177814107.xyz: 98 atoms
177814716.xyz: 98 atoms
177810456.xyz: 106 atoms
177820451.xyz: 114 atoms
177810731.xyz: 114 atoms
177808434.xyz: 114 atoms
177808447.xyz: 116 atoms
177807328.xyz: 116 atoms
177807354.xyz: 119 atoms
177807340.xyz: 122 atoms
177810565.xyz: 126 atoms
177808224.xyz: 12

In [ ]:
loaded_xyz_structures[0]

Atoms(symbols='C4NCOCOC4NCOCOH26', pbc=False)

In [51]:
sccalculator.model.model.max_radius = 1000
sccalculator.model.model.max_num_neighbors = 200

In [68]:
idx = 0
start_time = time.time()
sccalculator.calculate(loaded_xyz_structures[idx], properties=["energy", "forces"], system_changes=["positions"],filtering=True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(sccalculator.results)

Time taken: 40.64167785644531 seconds
{'energy': np.float64(-771.1453517994723), 'forces': array([[-8.50380717e+00,  5.25728329e+01,  1.41783442e+01],
       [-1.10007175e+02,  9.24846364e+01,  5.48604953e+01],
       [-1.17370630e+01, -2.73512769e+01,  1.47565732e+02],
       [ 1.08444559e+01,  3.41660592e+01, -5.42973091e+01],
       [-3.78365073e+00,  8.72520949e+00,  4.12637871e+00],
       [ 1.19839629e+01, -4.13946476e+01, -2.76962691e+00],
       [-2.94464288e+00, -1.79728743e+01, -2.11059258e+01],
       [-5.86268432e+00, -4.86572570e+01, -9.27694802e+00],
       [-6.63474742e+00, -1.01976725e+00, -2.04803908e+01],
       [-2.86869940e+01, -4.77848472e+01, -4.19880972e+00],
       [ 8.03823867e+01, -9.61504466e+01,  6.93711571e+01],
       [ 2.59912808e+01,  4.71657187e+00, -1.79352002e+02],
       [ 6.42073253e+01, -5.09482522e+01,  7.16720432e+01],
       [-1.77683444e+01, -8.51426208e-01, -2.83464184e+01],
       [ 7.00940936e+01,  4.70366954e+01,  8.12340231e+00],
       [-

In [82]:
from md.scflow_calculator_gpu import SCFlowRKSCalculator, RKSCalculator
sccalculator = SCFlowRKSCalculator(basis="def2-SVP", functional="b3lyp")
sccalculator.set_model(cur_ckpt, data_type="qh9", units="ang", model_length_unit="ang", mf_init_functional="b3lyp",)

INFO     [__init__.py:104] >> model_args: {'in_node_features': 1, 'sh_lmax': 4, 'hidden_size': 256,                
         'bottle_hidden_size': 64, 'num_gnn_layers': 4, 'max_radius': 15, 'num_nodes': 10, 'radius_embed_dim': 16, 
         'max_T': 15, 'use_block_S': True, 'use_block_H': True, 'ham_dim': 24, 'ham_hidden': 288, 'dataset_type':  
         'qh9', 'num_ham_gnn_layers': 2}

Setting device to cuda


INFO     [base_module.py:128] >> use_init_hamiltonian: True

INFO     [base_module.py:129] >> use_init_hamiltonian_residue: True

INFO     [base_module.py:130] >> ema_start_epoch: -1

INFO     [base_module.py:131] >> qh9: True

INFO     [base_module.py:132] >> test_mode: test

INFO     [flow_module.py:211] >> init_p0_type: expand

model trained on dataset:  QH9Stable
ode_steps: 1


In [96]:
sccalculator.set_model(
    cur_ckpt, 
    data_type="qh9", 
    units="ang", 
    model_length_unit="ang", 
    mf_init_functional="b3lyp",
    filtering=True,
    gt_tol=1e-8,
    pred_tol=1e-3,
    pad_eigval=1,
)

INFO     [__init__.py:104] >> model_args: {'in_node_features': 1, 'sh_lmax': 4, 'hidden_size': 256,                
         'bottle_hidden_size': 64, 'num_gnn_layers': 4, 'max_radius': 15, 'num_nodes': 10, 'radius_embed_dim': 16, 
         'max_T': 15, 'use_block_S': True, 'use_block_H': True, 'ham_dim': 24, 'ham_hidden': 288, 'dataset_type':  
         'qh9', 'num_ham_gnn_layers': 2}

Setting device to cuda


INFO     [base_module.py:128] >> use_init_hamiltonian: True

INFO     [base_module.py:129] >> use_init_hamiltonian_residue: True

INFO     [base_module.py:130] >> ema_start_epoch: -1

INFO     [base_module.py:131] >> qh9: True

INFO     [base_module.py:132] >> test_mode: test

INFO     [flow_module.py:211] >> init_p0_type: expand

model trained on dataset:  QH9Stable
ode_steps: 1


In [98]:
loaded_xyz_structures[idx]

Atoms(symbols='C4OCONC9OCF3CO2H20', pbc=False)

In [97]:
idx = 1
start_time = time.time()
sccalculator.calculate(loaded_xyz_structures[idx], properties=["energy", "forces"], system_changes=["positions"])
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(sccalculator.results)

Time taken: 20.29735565185547 seconds
{'energy': np.float64(-1351.1924839340895), 'forces': array([[ 0.00625802, -0.00473452, -0.0003973 ],
       [-0.01611398,  0.00394085, -0.00552475],
       [ 0.0083974 , -0.00093444,  0.00523647],
       [ 0.00736974, -0.00282924, -0.00141932],
       [ 0.00900054, -0.00305222,  0.00108375],
       [ 0.00320741, -0.00373659,  0.00073462],
       [ 0.00148413,  0.00298299,  0.0075093 ],
       [-0.00864269, -0.00075382, -0.01281992],
       [ 0.00258154, -0.00445711, -0.00311403],
       [-0.02450454, -0.00214642, -0.00501679],
       [ 0.01439807,  0.00253982, -0.00267295],
       [-0.01079126, -0.00757143, -0.00381298],
       [ 0.0064158 ,  0.00496605,  0.0025346 ],
       [ 0.00150959, -0.00097916, -0.00080019],
       [-0.00326494, -0.00070366, -0.00038232],
       [ 0.00458131, -0.00160448, -0.00032043],
       [ 0.01313172, -0.02250427, -0.00984756],
       [ 0.00709774,  0.05171468,  0.02431093],
       [-0.01506453, -0.00821448, -0.0055462

In [95]:
idx = 1
start_time = time.time()
sccalculator.calculate(loaded_xyz_structures[idx], properties=["energy", "forces"], system_changes=["positions"],filtering=True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(sccalculator.results)

Time taken: 25.038153171539307 seconds
{'energy': np.float64(-1351.192483943205), 'forces': array([[ 0.00625797, -0.00473449, -0.0003973 ],
       [-0.01611398,  0.00394075, -0.00552458],
       [ 0.0083974 , -0.00093444,  0.00523648],
       [ 0.00736969, -0.00282923, -0.00141929],
       [ 0.00900061, -0.00305226,  0.00108371],
       [ 0.00320737, -0.0037367 ,  0.0007344 ],
       [ 0.00148408,  0.0029831 ,  0.00750944],
       [-0.00864272, -0.00075373, -0.01281979],
       [ 0.00258146, -0.0044571 , -0.00311386],
       [-0.0245047 , -0.00214648, -0.0050169 ],
       [ 0.0143983 ,  0.00253986, -0.00267275],
       [-0.01079099, -0.0075715 , -0.00381322],
       [ 0.00641577,  0.004966  ,  0.00253473],
       [ 0.00150977, -0.00097919, -0.0008002 ],
       [-0.00326491, -0.00070351, -0.00038227],
       [ 0.00458134, -0.00160455, -0.00032046],
       [ 0.01313185, -0.02250432, -0.00984746],
       [ 0.00709747,  0.05171448,  0.02431083],
       [-0.01506411, -0.00821424, -0.0055461

In [100]:
start_time = time.time()
rkscalculator.calculate(loaded_xyz_structures[idx], properties=["energy", "forces"], system_changes=["positions"])
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(rkscalculator.results)

Time taken: 30.066532611846924 seconds
{'energy': np.float64(-1351.1967818631615), 'forces': array([[ 8.09169139e-03, -3.60909529e-03, -2.73469303e-03],
       [-1.67337653e-02,  3.33151013e-03, -7.28300179e-03],
       [ 7.79034607e-03,  8.79326759e-04,  7.28936270e-03],
       [ 7.22927301e-03, -5.70598361e-03, -1.45674904e-03],
       [ 8.32740144e-03, -2.16996109e-03,  2.07938351e-03],
       [ 5.07494268e-03, -7.13179951e-03, -3.78600459e-03],
       [ 9.14956261e-04,  5.21717233e-03,  1.09266088e-02],
       [-7.73668247e-03, -5.63030061e-04, -1.01344933e-02],
       [ 3.87635075e-03, -3.70943524e-03, -1.56417719e-03],
       [-2.32958526e-02,  3.24019508e-03, -5.30371636e-03],
       [ 1.51535732e-02,  3.31073614e-03, -6.52461981e-03],
       [-8.51122303e-03, -8.30961464e-03, -3.73820938e-03],
       [ 4.80873307e-03,  6.74528178e-03,  2.60539410e-03],
       [ 1.21793175e-03, -1.37530786e-03, -7.23057335e-04],
       [-5.28425548e-03, -5.77346562e-04, -5.08405201e-04],
       